In [7]:
# 先来导入库
import torch
import torch.nn as nn
from torch.utils.data import Dataset
# names_train.csv.gz是压缩文件，普通的open（）打不开
# 所以用gzip.open()
import gzip
# 读取csv
import csv
# 导入string 包，为后面将人名拆开做准备
import string
import random
# 创建一个 字符字典表
# 规定将所有可能出现的字符都放入这个名单里面，包含大小写和特殊字符
all_letters = string.ascii_letters + " .,;'"

# 输入一个字符，返回它在字符表中的位置
# 方便变成tensor 和后续的Embedding处理
def letterToIndex(letter):
    return all_letters.find(letter)

# 定义将整个名字转化为Tensor函数
def lineToTensor(line):
    # 创建空列表,等会保存遍历出来的位置
    tensor= []

    # 遍历每一个字符
    for letter in line:
        # 编码
        index = letterToIndex(letter)
        tensor.append(index)
        # 将list 变成 torch.tensor
    return torch.tensor(tensor)

# 开始写dataset
class NameDataset(Dataset):
    def __init__(self,is_train_set = True):

        filename = 'data/names_train.csv.gz' if is_train_set else 'data/names_test.csv.gz'
        with gzip.open(filename,'rt') as f :
            # 这里的f指的是我们打开的文件夹
            # 如果不用reader则读取的只是字符串 line = "Mary,English"
            # 经过reader = csv.reader(f) 则变成['Mary','English']
            reader = csv.reader(f)

            # 这个是csv读取器对象，读取完后创建一个二维列表
            # 也就是说 csv.reader 负责 文件一行一行解析-> 变成列表格式
            # list() 属于迭代器 -> 全部取出来 -> 保存
            rows = list(reader)

        self.names = [row[0] for row in rows]
        self.countries = [row[1] for row in rows]

        # 开始处理国家类型

        # 1.找出所有国家
        # set()去重 -> sorted 进行排序 -> list 因为sort返回的是列表，所以这里要保证list类型
        self.country_list = list(sorted(set(self.countries)))

        # 2.创建一个 country_dict 字典 比如 Arabic -> 0 ,Chinese ->1
        # 进行 Label Encoding (标签编码)
        # 创建空字典
        self.country_dict = {}
        # 这里用enumerate 等价于循环时 idx = i,country = ...
        for idx,country in enumerate(self.country_list):
            # 放入字典
            self.country_dict[country] = idx

    # 下面来实现 len 和 getitem 这两个核心函数
    def __len__(self):
        return len(self.names)

    def __getitem__(self,index):
        name = self.names[index]
        country = self.countries[index]
        # 将名字转为 Tensor
        name_tensor = lineToTensor(name)
        # 将国家转为数字
        country_index = self.country_dict[country]

        return name_tensor,country_index



dataset = NameDataset()

# 这里不能直接创建 DataLoader 因为人名名字可能不同
# 但是神经网络要求【batch_size,seq_len】,所以这里需要 padding
# 由于这个数据集比较小，所以这里决定单个单个处理，测试集也是
# 定义模型
class BiLSTMClassifier(nn.Module):
    # input_size 不是输入名字长度，也不是batch_size,是字符种类数量
    # hidden_size 是RNN隐藏层大小
    # output_size 指最后分类数量
    def __init__(self,input_size,hidden_size,output_size):
        super().__init__()

        # 定义Embedding
        # 建立 57 个字符 对应 128 维向量
        self.embedding =nn.Embedding(
            input_size,
            hidden_size
        )

        # 定义RNN
        # 两个hidden_size 因为Embedding输出128维，RNN输入128维
        # 这里没写 num_layers 默认了num_layers = 1
        # 这里加上bidirectional = true ,把lstm 变成 bilstm
        # 加上后output的最后一维变化了
        # 因为此时有 forward hidden + backward hidden
        # hidden 和 cell 的最后一维也变化了，因为要加上正向和反向的情况

        # BiLSTM
        self.lstm = nn.LSTM(hidden_size,hidden_size,batch_first=True,bidirectional=True)

        self.fc = nn.Linear(hidden_size * 2,output_size)
    def forward(self,x):
        # 假如输入aaaaa 经过embedding后由【5】变成【5，128】
        # 这里的5相当于5个时间步，而embedding建了一个【57，128】的表，本质是一个查表操作
        # 意思如果是【0，0，0，0，0】，取embedding第0行128个数字

        # 也就是 seq_len = 5,embedding_dim = 128
        x = self.embedding(x)
        # 增加batch维度，现在【5，128】 代表【seq_len,embedding_dim】
        # 但是我们定义的rnn 里面有batch_first = True
        # 所以RNN 默认希望维度为【batch,seq,feature】

        # 这里意思是在第0维加入一个batch维度
        x = x.unsqueeze(0)

        # output保存hidden1,hidden2.... shape为[1，5，128]
        # hidden表示最后一个隐藏状态(最后记忆) shape为[1,1,128]
        # hidden 表示最后隐藏状态，而cell表示最后记忆状态
        output,(hidden,cell) = self.lstm(x)
        # 现在hidden是[2,1,128] 但是linear 要【batch,feature】
        # 此时的【2，1，128】 为 【num_layers,batch_size,hidden_size】
        # 由于第一维是2，所以不能直接用squeeze删去
        # 得先将 forward hidden 和 backward hidden 合并成一个hidden
        # 即 【1，1，128】 + 【1，1，128】 ->[1,1,256]

        # 这里取hidden[0],hidden[1] 已经完成了删去第一维 num_layer的操作了 todo
        # 取正向最后的hidden
        # hidden [2,1,64]
        # hidden[-2] forward最后的状态
        # hidden[-1] backward最后的状态
        forward_hidden = hidden[-2]
        # 取反向最后的hidden
        backward_hidden = hidden[-1]

        # 拼接两个方向
        hidden = torch.cat([forward_hidden,backward_hidden],dim=1)
        output = self.fc(hidden)
        return output
model = BiLSTMClassifier(
    len(all_letters),
    64,
    len(dataset.country_list)
)
name_tensor = lineToTensor("Smith")

output = model(name_tensor)

print(output.shape)
# 定义损失函数和优化器
criterion = nn.CrossEntropyLoss()
optimizer = torch.optim.Adam(model.parameters(),lr= 0.0005)

# 加载测试集
test_dataset = NameDataset(False)
epochs = 10
best_acc = 0

for epoch in range(epochs):

    # =====================
    # 训练
    # =====================

    total_loss = 0

    indices = list(range(len(dataset)))
    random.shuffle(indices)


    for i in indices:

        name_tensor,label = dataset[i]

        output = model(name_tensor)

        label = torch.tensor([label])

        loss = criterion(
            output,
            label
        )


        optimizer.zero_grad()

        loss.backward()

        optimizer.step()


        total_loss += loss.item()



    avg_loss = total_loss / len(dataset)



    # =====================
    # 测试
    # =====================

    correct = 0
    total = 0


    with torch.no_grad():

        for name_tensor,label in test_dataset:

            output = model(name_tensor)

            pred = output.argmax(dim=1).item()


            if pred == label:
                correct += 1

            total += 1



    acc = correct / total



    print(
        f"Epoch [{epoch+1}/{epochs}], "
        f"Loss:{avg_loss:.4f}, "
        f"Accuracy:{acc:.4f}"
    )



    # =====================
    # 保存最佳模型
    # =====================

    if acc > best_acc:

        best_acc = acc

        torch.save(
            model.state_dict(),
            "best_rnn_name_classifier.pth"
        )

        print("保存最佳模型")

# 加载最佳模型
model.load_state_dict(
    torch.load(
        "best_rnn_name_classifier.pth"
    )
)
# 写预测函数
def predict(name):
    # 1.字符串转tensor
    name_tensor = lineToTensor(name)

    # 2.关闭梯度
    with torch.no_grad():
        output = model(name_tensor)

    pred = output.argmax(dim=1).item()
    # 3.数字转国家
    country = dataset.country_list[pred]

    return country
print(predict("Lizhuofan"))
print(predict("lizhuofan"))
print(predict("Jiangxinrui"))
print(predict("konglong"))
print(predict("Curry"))

torch.Size([1, 18])
Epoch [1/10], Loss:0.9389, Accuracy:0.7896
保存最佳模型
Epoch [2/10], Loss:0.6057, Accuracy:0.8125
保存最佳模型
Epoch [3/10], Loss:0.4916, Accuracy:0.8327
保存最佳模型
Epoch [4/10], Loss:0.4141, Accuracy:0.8293
Epoch [5/10], Loss:0.3526, Accuracy:0.8349
保存最佳模型
Epoch [6/10], Loss:0.3005, Accuracy:0.8387
保存最佳模型
Epoch [7/10], Loss:0.2519, Accuracy:0.8355
Epoch [8/10], Loss:0.2151, Accuracy:0.8372
Epoch [9/10], Loss:0.1865, Accuracy:0.8364
Epoch [10/10], Loss:0.1623, Accuracy:0.8406
保存最佳模型
Russian
Russian
Russian
English
English
